# STAT3612: Multimodal Brain Tumor Classification — Full Pipeline

This notebook implements the **three-stage pipeline** described in our proposal:

| Section | Proposal Stage | Content |
|---|---|---|
| **0. Setup & EDA** | Preliminary Findings | Environment, data loading, EDA findings (class imbalance, missing data, multimodal incompleteness) |
| **1. Feature Engineering** | Data Processing | Load & encode all 4 modalities; PCA; standardize |
| **2. Single-Modality Baselines** | Stage 1 | Independent classifiers per modality to understand individual contributions |
| **3. Multimodal Fusion** | Stage 2 | Early fusion (concatenated features) and late fusion (voting/stacking) |
| **4. Optimization** | Stage 3 | Class imbalance mitigation, feature selection *(TODO)* |
| **5. Model Analysis** | Analysis Plan | Model comparison, modality ablation, interpretability |
| **6. Kaggle Submission** | — | Generate final submission file |

**Evaluation metric**: Macro-F1  
**5-class target**: Glioma, Meningioma, Brain Metastase Tumour, Tumors of the sellar region, Pineal tumour and Choroid plexus tumour

---
# 0. Setup & Exploratory Data Analysis

## 0.1 Environment

In [ ]:
import numpy as np
import pandas as pd
import json, os, warnings
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

def _find_data_root() -> Path:
    for d in [Path.cwd(), *Path.cwd().resolve().parents]:
        if (d / 'kaggle-dataset' / 'train.json').is_file():
            return d / 'kaggle-dataset'
    raise FileNotFoundError('Cannot locate kaggle-dataset/train.json')

BASE_DIR = str(_find_data_root())
print('BASE_DIR =', BASE_DIR)

## 0.2 Load Metadata

In [ ]:
with open(os.path.join(BASE_DIR, 'train.json')) as f:
    train_meta = json.load(f)
with open(os.path.join(BASE_DIR, 'val.json')) as f:
    val_meta = json.load(f)
with open(os.path.join(BASE_DIR, 'test.json')) as f:
    test_meta = json.load(f)

print(f'Splits — Train: {len(train_meta)}, Val: {len(val_meta)}, Test: {len(test_meta)}')

train_labels = {k: v['Overall_class'] for k, v in train_meta.items()}
val_labels   = {k: v['Overall_class'] for k, v in val_meta.items()}

## 0.3 EDA: Class Imbalance (~40:1)

In [ ]:
label_counts = pd.Series(Counter(train_labels.values())).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, ax=ax)
ax.set_title('Training Label Distribution (5-class)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

print('Class frequencies:')
for cls, cnt in label_counts.items():
    print(f'  {cls}: {cnt} ({cnt/len(train_labels)*100:.1f}%)')
print(f'\nImbalance ratio (max/min): {label_counts.max()/label_counts.min():.0f}:1')

## 0.4 EDA: Multimodal Incompleteness (~20% missing 1 modality)

In [ ]:
MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']

for split_name, meta in [('train', train_meta), ('val', val_meta), ('test', test_meta)]:
    n_paths = [len(v.get('image_path', [])) for v in meta.values()]
    counts = Counter(n_paths)
    total = len(meta)
    print(f'[{split_name}] modality completeness: ', end='')
    for k in sorted(counts):
        print(f'{k}-mod={counts[k]} ({counts[k]/total*100:.1f}%)  ', end='')
    print()

## 0.5 EDA: Missing Demographics (JSON ~70% vs CSV ~0%)

In [ ]:
def _clean_cols(df):
    df.columns = [str(c).replace(chr(0xFEFF), '').strip() for c in df.columns]
    return df

clin_train = _clean_cols(pd.read_csv(os.path.join(BASE_DIR, 'clinical_information', 'train_patient_info.csv')))

json_age_missing = sum(1 for v in train_meta.values() if not v.get('Age'))
json_sex_missing = sum(1 for v in train_meta.values() if not v.get('Sex'))
clin_age_missing = clin_train['Age'].isna().sum()
clin_sex_missing = clin_train['Sex'].isna().sum()

print(f'JSON  Age missing: {json_age_missing}/{len(train_meta)} ({json_age_missing/len(train_meta)*100:.1f}%)')
print(f'JSON  Sex missing: {json_sex_missing}/{len(train_meta)} ({json_sex_missing/len(train_meta)*100:.1f}%)')
print(f'CSV   Age missing: {clin_age_missing}/{len(clin_train)} ({clin_age_missing/len(clin_train)*100:.1f}%)')
print(f'CSV   Sex missing: {clin_sex_missing}/{len(clin_train)} ({clin_sex_missing/len(clin_train)*100:.1f}%)')
print('\n=> Clinical CSV is the preferred source for demographics.')

## 0.6 EDA: Radiomics Feature Correlation

In [ ]:
RADIOMICS_FEAT_COLS = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                       'rad_firstorder_90Percentile', 'rad_glcm_Contrast',
                       'rad_glcm_JointEntropy']

rad_frames = []
for mod in MODALITIES:
    df = _clean_cols(pd.read_csv(os.path.join(BASE_DIR, 'radiomics_info', 'train', f'{mod}_radiomics_train.csv')))
    for c in RADIOMICS_FEAT_COLS:
        df = df.rename(columns={c: f'{mod}_{c}'})
    rad_frames.append(df)

rad_all = rad_frames[0][['case_id']].copy()
for df in rad_frames:
    rad_all = rad_all.merge(df.drop(columns=['sex','age','modality'], errors='ignore'), on='case_id', how='left')

rad_numeric = rad_all.select_dtypes(include=[np.number]).drop(columns=['case_id'], errors='ignore')
corr = rad_numeric.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, xticklabels=True, yticklabels=True)
plt.title('Radiomics Feature Correlation (Train)')
plt.tight_layout(); plt.show()

high_corr = (corr.abs() > 0.95).sum().sum() - len(corr)
print(f'Feature pairs with |r| > 0.95: {high_corr // 2}')

---
# 1. Feature Engineering

We load and encode all four modalities, then combine them into a unified feature matrix.

## 1.1 Image Features (2048-d × 4 modalities, zero-pad if missing)

In [ ]:
def load_image_features(meta_dict, base_dir=BASE_DIR):
    case_ids = sorted(meta_dict.keys(), key=int)
    features = []
    for cid in case_ids:
        case_feats = []
        for mod in MODALITIES:
            npy_path = os.path.join(base_dir, 'image_features', 'image_features', cid, mod, 'image.npy')
            if os.path.exists(npy_path):
                case_feats.append(np.load(npy_path))
            else:
                case_feats.append(np.zeros(2048, dtype=np.float32))
        features.append(np.concatenate(case_feats))
    return np.array(features), case_ids

X_img_train, train_ids = load_image_features(train_meta)
X_img_val,   val_ids   = load_image_features(val_meta)
X_img_test,  test_ids  = load_image_features(test_meta)
print(f'Image — Train: {X_img_train.shape}, Val: {X_img_val.shape}, Test: {X_img_test.shape}')

## 1.2 Radiomics Features (5 × 4 modalities, NaN → 0)

In [ ]:
def load_radiomics(split, case_ids, base_dir=BASE_DIR):
    all_feats = pd.DataFrame({'case_id': [int(c) for c in case_ids]})
    for mod in MODALITIES:
        df = _clean_cols(pd.read_csv(os.path.join(base_dir, 'radiomics_info', split, f'{mod}_radiomics_{split}.csv')))
        rename = {c: f'{mod}_{c}' for c in RADIOMICS_FEAT_COLS}
        df = df[['case_id'] + RADIOMICS_FEAT_COLS].rename(columns=rename)
        all_feats = all_feats.merge(df, on='case_id', how='left')
    feat_cols = [c for c in all_feats.columns if c != 'case_id']
    all_feats[feat_cols] = all_feats[feat_cols].fillna(0)
    all_feats = all_feats.set_index('case_id').loc[[int(c) for c in case_ids]]
    return all_feats.values

X_rad_train = load_radiomics('train', train_ids)
X_rad_val   = load_radiomics('val',   val_ids)
X_rad_test  = load_radiomics('test',  test_ids)
print(f'Radiomics — Train: {X_rad_train.shape}, Val: {X_rad_val.shape}, Test: {X_rad_test.shape}')

## 1.3 Clinical Features (from CSV; demographics sourced from clinical_information)

In [ ]:
def load_clinical(split, case_ids, base_dir=BASE_DIR):
    df = _clean_cols(pd.read_csv(os.path.join(base_dir, 'clinical_information', f'{split}_patient_info.csv')))
    df = df.set_index('case_id').loc[[int(c) for c in case_ids]].reset_index()
    return df

df_clin_train = load_clinical('train', train_ids)
df_clin_val   = load_clinical('val',   val_ids)
df_clin_test  = load_clinical('test',  test_ids)

sex_map = {'female': 0, 'male': 1, 'unknown': 2}
for df in [df_clin_train, df_clin_val, df_clin_test]:
    df['Sex_enc'] = df['Sex'].map(sex_map)

train_age_median = df_clin_train['Age'].median()
for df in [df_clin_train, df_clin_val, df_clin_test]:
    df['Age_filled'] = df['Age'].fillna(train_age_median)

intensity_map = {'hypointense': 0, 'isointense': 1, 'hyperintense': 2,
                 'heterogeneous': 3, 'homogeneous': 4, 'unknown': 5}
intensity_cols = ['Signal Intensity (T1)', 'Signal Intensity (T1c)',
                  'Signal Intensity (T2)', 'Signal Intensity (T2-FLAIR)']
for df in [df_clin_train, df_clin_val, df_clin_test]:
    for col in intensity_cols:
        df[col + '_enc'] = df[col].map(intensity_map).fillna(5).astype(int)

location_keywords = ['frontal', 'temporal', 'parietal', 'occipital', 'cerebellum',
                     'sellar', 'sella', 'pituitary', 'pineal', 'ventricle',
                     'brainstem', 'cerebellopontine', 'thalamus', 'basal',
                     'left', 'right', 'bilateral', 'midline']
for df in [df_clin_train, df_clin_val, df_clin_test]:
    loc_lower = df['Tumor Location'].str.lower().fillna('')
    for kw in location_keywords:
        df[f'loc_{kw}'] = loc_lower.str.contains(kw).astype(int)

clinical_feat_cols = (['Sex_enc', 'Age_filled'] +
                      [c + '_enc' for c in intensity_cols] +
                      [f'loc_{kw}' for kw in location_keywords])

X_clin_train = df_clin_train[clinical_feat_cols].values.astype(float)
X_clin_val   = df_clin_val[clinical_feat_cols].values.astype(float)
X_clin_test  = df_clin_test[clinical_feat_cols].values.astype(float)
print(f'Clinical — Train: {X_clin_train.shape}, Val: {X_clin_val.shape}, Test: {X_clin_test.shape}')

## 1.4 Text Features (TF-IDF from radiology reports)

In [ ]:
def get_reports(meta_dict, case_ids):
    return [meta_dict[c].get('report', '') or '' for c in case_ids]

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english', min_df=3)
X_text_train = tfidf.fit_transform(get_reports(train_meta, train_ids)).toarray()
X_text_val   = tfidf.transform(get_reports(val_meta, val_ids)).toarray()
X_text_test  = tfidf.transform(get_reports(test_meta, test_ids)).toarray()
print(f'Text — Train: {X_text_train.shape}, Val: {X_text_val.shape}, Test: {X_text_test.shape}')

## 1.5 PCA, Standardize, Encode Labels

In [ ]:
pca_img = PCA(n_components=256, random_state=SEED)
X_img_train_pca = pca_img.fit_transform(X_img_train)
X_img_val_pca   = pca_img.transform(X_img_val)
X_img_test_pca  = pca_img.transform(X_img_test)
print(f'PCA explained variance (256 comp): {pca_img.explained_variance_ratio_.sum():.4f}')

# Early-fusion feature matrix
X_train_all = np.hstack([X_img_train_pca, X_rad_train, X_clin_train, X_text_train])
X_val_all   = np.hstack([X_img_val_pca,   X_rad_val,   X_clin_val,   X_text_val])
X_test_all  = np.hstack([X_img_test_pca,  X_rad_test,  X_clin_test,  X_text_test])
print(f'Early-fusion features: {X_train_all.shape[1]}d (img 256 + rad {X_rad_train.shape[1]} + clin {X_clin_train.shape[1]} + text {X_text_train.shape[1]})')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_all)
X_val_scaled   = scaler.transform(X_val_all)
X_test_scaled  = scaler.transform(X_test_all)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform([train_labels[c] for c in train_ids])
y_val   = label_encoder.transform([val_labels[c] for c in val_ids])
print(f'Labels: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}')

# Class weights for imbalance handling
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))
sample_weights_train = np.array([class_weight_dict[y] for y in y_train])
for idx, w in class_weight_dict.items():
    print(f'  {label_encoder.inverse_transform([idx])[0]}: weight={w:.3f}')

---
# 2. Stage 1 — Single-Modality Baselines

Before fusion, we evaluate each modality independently to understand its discriminative power. This directly informs which modalities are worth fusing and which fusion strategy to use.

In [ ]:
def eval_modality(X_tr, X_va, y_tr, y_va, name, model=None):
    """Standardize, train, evaluate a single modality block."""
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    if model is None:
        model = lgb.LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                                   class_weight='balanced', random_state=SEED, verbose=-1)
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_va_s)
    f1 = f1_score(y_va, y_pred, average='macro')
    print(f'  {name:25s} | Macro-F1 = {f1:.4f}')
    return f1, model, sc

print('Single-Modality Baselines (LightGBM):')
single_results = {}
single_models = {}
single_scalers = {}

for mod_name, Xtr, Xva in [
    ('Image (PCA-256)',  X_img_train_pca, X_img_val_pca),
    ('Radiomics (20-d)', X_rad_train,     X_rad_val),
    ('Clinical (24-d)',  X_clin_train,    X_clin_val),
    ('Text (TF-IDF 500)', X_text_train,  X_text_val),
]:
    f1, mdl, sc = eval_modality(Xtr, Xva, y_train, y_val, mod_name)
    single_results[mod_name] = f1
    single_models[mod_name] = mdl
    single_scalers[mod_name] = sc

---
# 3. Stage 2 — Multimodal Fusion

We compare **early fusion** (concatenated features → single model) and **late fusion** (combine single-modality predictions).

## 3.1 Early Fusion — Model Comparison

Using the 800-d concatenated feature set, we train models of increasing flexibility.

In [ ]:
def evaluate_model(model, X, y, name='Model'):
    y_pred = model.predict(X)
    f1m = f1_score(y, y_pred, average='macro')
    print(f'\n{"="*55}\n{name}  |  Macro-F1 = {f1m:.4f}\n{"="*55}')
    print(classification_report(y, y_pred, target_names=label_encoder.classes_))
    return f1m

early_results = {}

In [ ]:
# 3.1.1 Logistic Regression (L2) — linear baseline
lr = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced',
                        multi_class='multinomial', solver='lbfgs', random_state=SEED)
lr.fit(X_train_scaled, y_train)
early_results['LR (L2)'] = evaluate_model(lr, X_val_scaled, y_val, 'Logistic Regression (L2)')

In [ ]:
# 3.1.2 Random Forest — bagging ensemble
rf = RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=2,
                            class_weight='balanced', random_state=SEED, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
early_results['RF'] = evaluate_model(rf, X_val_scaled, y_val, 'Random Forest')

In [ ]:
# 3.1.3 SVM (RBF kernel)
svm = SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced',
          random_state=SEED, probability=True)
svm.fit(X_train_scaled, y_train)
early_results['SVM-RBF'] = evaluate_model(svm, X_val_scaled, y_val, 'SVM (RBF)')

In [ ]:
# 3.1.4 XGBoost — boosting ensemble
xgb_model = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, objective='multi:softprob', num_class=5,
    eval_metric='mlogloss', random_state=SEED, use_label_encoder=False)
xgb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights_train)
early_results['XGB'] = evaluate_model(xgb_model, X_val_scaled, y_val, 'XGBoost')

In [ ]:
# 3.1.5 LightGBM
lgb_model = lgb.LGBMClassifier(n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, num_leaves=63,
    class_weight='balanced', random_state=SEED, verbose=-1)
lgb_model.fit(X_train_scaled, y_train)
early_results['LGB'] = evaluate_model(lgb_model, X_val_scaled, y_val, 'LightGBM')

In [ ]:
# 3.1.6 MLP Neural Network
mlp = MLPClassifier(hidden_layer_sizes=(512, 256, 128), activation='relu',
    max_iter=500, early_stopping=True, validation_fraction=0.1,
    batch_size=64, learning_rate='adaptive', learning_rate_init=0.001, random_state=SEED)
mlp.fit(X_train_scaled, y_train)
early_results['MLP'] = evaluate_model(mlp, X_val_scaled, y_val, 'MLP')

## 3.2 Late Fusion — Voting & Stacking

Combine predictions from individually strong models. This preserves modality-specific structure.

In [ ]:
late_results = {}

# 3.2.1 Soft Voting (early-fusion models combined by probability averaging)
voting_clf = VotingClassifier(estimators=[
    ('svm', SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced', random_state=SEED, probability=True)),
    ('xgb', xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED, use_label_encoder=False, eval_metric='mlogloss')),
    ('lgb', lgb.LGBMClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, num_leaves=63, class_weight='balanced', random_state=SEED, verbose=-1)),
    ('lr', LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', multi_class='multinomial', random_state=SEED)),
], voting='soft')
voting_clf.fit(X_train_scaled, y_train)
late_results['Soft Voting'] = evaluate_model(voting_clf, X_val_scaled, y_val, 'Late Fusion: Soft Voting')

In [ ]:
# 3.2.2 Stacking (meta-learner: LR on base model outputs, 5-fold CV)
stacking_clf = StackingClassifier(estimators=[
    ('svm', SVC(C=10, kernel='rbf', gamma='scale', class_weight='balanced', random_state=SEED, probability=True)),
    ('xgb', xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED, use_label_encoder=False, eval_metric='mlogloss')),
    ('lgb', lgb.LGBMClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, num_leaves=63, class_weight='balanced', random_state=SEED, verbose=-1)),
    ('rf', RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=2, class_weight='balanced', random_state=SEED)),
], final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED), cv=5)
stacking_clf.fit(X_train_scaled, y_train)
late_results['Stacking'] = evaluate_model(stacking_clf, X_val_scaled, y_val, 'Late Fusion: Stacking')

---
# 4. Stage 3 — Optimization *(TODO sections for team)*

The cells below are placeholders for experiments to be completed during Apr 9–13.

### 4.1 Feature Selection
- Compare LASSO (L1) vs RF importance vs MI ranking
- Remove radiomics pairs with |r| > 0.95
- Tune PCA components (128 / 256 / 512)

### 4.2 Class Imbalance Experiments
- (a) class_weight='balanced' (current default)
- (b) Random oversampling of minority classes (train only)
- (c) SMOTE in PCA-reduced feature space (train only)
- Report per-class F1 and macro-F1 for each strategy

### 4.3 Hyperparameter Tuning
- Stratified 5-fold CV on train set for top-2 models
- Grid search over key parameters (e.g., LGB n_estimators, max_depth, learning_rate)

---
# 5. Model Analysis

Targeting the workload/analysis component (40% of grade).

## 5.1 Model Comparison Summary (Early + Late Fusion)

In [ ]:
all_results = {**early_results, **late_results}

print('\n' + '='*60)
print('FULL MODEL COMPARISON (Validation Macro-F1)')
print('='*60)
for name, f1 in sorted(all_results.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(f1 * 50)
    print(f'{name:15s} | {f1:.4f} | {bar}')
best_name = max(all_results, key=all_results.get)
print(f'\n*** Best: {best_name} (Macro-F1 = {all_results[best_name]:.4f}) ***')

## 5.2 Modality Ablation Study

In [ ]:
feature_groups = {
    'Image Only':       (X_img_train_pca, X_img_val_pca),
    'Radiomics Only':   (X_rad_train, X_rad_val),
    'Clinical Only':    (X_clin_train, X_clin_val),
    'Text Only':        (X_text_train, X_text_val),
    'Img+Rad':          (np.hstack([X_img_train_pca, X_rad_train]), np.hstack([X_img_val_pca, X_rad_val])),
    'Img+Clin':         (np.hstack([X_img_train_pca, X_clin_train]), np.hstack([X_img_val_pca, X_clin_val])),
    'Img+Text':         (np.hstack([X_img_train_pca, X_text_train]), np.hstack([X_img_val_pca, X_text_val])),
    'All (Early Fusion)': (X_train_all, X_val_all),
}

ablation = {}
for name, (Xtr, Xva) in feature_groups.items():
    sc = StandardScaler()
    m = lgb.LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                           class_weight='balanced', random_state=SEED, verbose=-1)
    m.fit(sc.fit_transform(Xtr), y_train)
    ablation[name] = f1_score(y_val, m.predict(sc.transform(Xva)), average='macro')

print('\nMODALITY ABLATION (LightGBM, Macro-F1)')
print('='*45)
for name, f1 in sorted(ablation.items(), key=lambda x: x[1], reverse=True):
    print(f'  {name:22s} | {f1:.4f}')

## 5.3 Interpretability *(TODO)*

To be completed during report writing phase (Apr 14–20):
- SHAP values on the best tree-based model
- Feature importance comparison: LASSO coefficients vs RF/XGBoost importance
- Confusion matrix analysis of the most confused class pairs

---
# 6. Generate Kaggle Submission

In [ ]:
model_map = {'LR (L2)': lr, 'RF': rf, 'SVM-RBF': svm, 'XGB': xgb_model,
             'LGB': lgb_model, 'MLP': mlp,
             'Soft Voting': voting_clf, 'Stacking': stacking_clf}
best_model = model_map[best_name]
print(f'Using best model: {best_name}')

y_test_pred = best_model.predict(X_test_scaled)
submission = pd.DataFrame({'case_id': [int(c) for c in test_ids],
                           'Overall_class': label_encoder.inverse_transform(y_test_pred)})
sample_sub = pd.read_csv(os.path.join(BASE_DIR, 'sample_submission.csv'))
submission = submission.set_index('case_id').loc[sample_sub['case_id']].reset_index()
submission.to_csv('submission.csv', index=False)

print(f'\nSubmission saved ({len(submission)} rows). Distribution:')
print(submission['Overall_class'].value_counts())